# 🧠 DermoViT — ISIC 2024 Training Notebook (Kaggle)

**Architecture:** DermoViT — Dual-Stream CNN + ViT with Novel ACAG Block + FiLM Metadata Conditioning

### How to run on Kaggle:
1. Add the ISIC 2024 dataset: `+ Add Data` → search **isic-2024-challenge**
2. Set accelerator to **GPU T4 x2** (or P100)
3. Run All Cells

**Dataset paths are pre-configured for Kaggle below.**


In [ ]:
# ─────────────────────────────────────────────
# CELL 1 — Install extra dependencies on Kaggle
# (timm and shap are not pre-installed)
# ─────────────────────────────────────────────
import subprocess
subprocess.run(['pip', 'install', 'timm', 'shap', '--quiet'], check=True)
print('✅ Dependencies installed')

In [ ]:
# ─────────────────────────────────────────────
# CELL 2 — Imports
# ─────────────────────────────────────────────
import os, io, gc, time, warnings
import h5py
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

import timm
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ─────────────────────────────────────────────
# CELL 3 — Kaggle Dataset Paths & Config
# ─────────────────────────────────────────────
DATA_DIR  = '/kaggle/input/isic-2024-challenge'
HDF5_PATH = os.path.join(DATA_DIR, 'train-image.hdf5')
META_PATH = os.path.join(DATA_DIR, 'train-metadata.csv')
OUT_DIR   = '/kaggle/working'
CKPT_PATH = os.path.join(OUT_DIR, 'dermovit_best.pth')

print(f'HDF5 exists:     {os.path.exists(HDF5_PATH)}')
print(f'Metadata exists: {os.path.exists(META_PATH)}')

# ── Hyperparameters ────────────────────────────
CFG = {
    'img_size':        224,
    'batch_size':      32,        # reduce to 16 if OOM
    'n_warmup':        3,         # epochs with frozen backbone
    'n_finetune':      7,         # epochs end-to-end
    'lr_warmup':       3e-4,
    'lr_finetune':     5e-5,
    'weight_decay':    1e-2,
    'focal_alpha':     0.25,
    'focal_gamma':     2.0,
    'label_smoothing': 0.05,
    'mixup_alpha':     0.4,
    'sam_rho':         0.05,
    'd_model':         512,
    'n_heads':         8,
    'asym_lambda':     0.5,
    'drop_rate':       0.3,
    'num_workers':     2,
    'fold':            0,         # which fold to train on
    'n_folds':         5,
    'seed':            42,
}

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

torch.manual_seed(CFG['seed'])
np.random.seed(CFG['seed'])
print('\nConfig loaded ✅')

In [ ]:
# ─────────────────────────────────────────────
# CELL 4 — Metadata Preprocessing
# ─────────────────────────────────────────────
df = pd.read_csv(META_PATH, low_memory=False)
print(f'Loaded metadata: {df.shape} | Malignant: {df["target"].sum():,} ({df["target"].mean()*100:.3f}%)')

FILM_NUMERIC = [
    'age_approx', 'clin_size_long_diam_mm',
    'tbp_lv_symm_2axis', 'tbp_lv_eccentricity', 'tbp_lv_norm_border',
    'tbp_lv_norm_color', 'tbp_lv_color_std_mean', 'tbp_lv_radial_color_std_max',
    'tbp_lv_nevi_confidence', 'tbp_lv_dnn_lesion_confidence',
    'tbp_lv_areaMM2', 'tbp_lv_area_perim_ratio',
]
FILM_CATEGORICAL = ['sex', 'anatom_site_general', 'tbp_tile_type']

for col in FILM_NUMERIC:
    df[col] = df[col].fillna(df[col].median())
for col in FILM_CATEGORICAL:
    df[col] = df[col].fillna('unknown')

df_cat   = pd.get_dummies(df[FILM_CATEGORICAL], drop_first=False)
META_COLS = FILM_NUMERIC + list(df_cat.columns)
df        = pd.concat([df, df_cat], axis=1)
META_DIM  = len(META_COLS)

# Patient-level stratified split (prevents patient leakage)
sgkf = StratifiedGroupKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=CFG['seed'])
folds = list(sgkf.split(df, df['target'], groups=df['patient_id']))
train_idx, val_idx = folds[CFG['fold']]

df_train = df.iloc[train_idx].copy().reset_index(drop=True)
df_val   = df.iloc[val_idx].copy().reset_index(drop=True)

# Scale numeric AFTER split (no leakage)
scaler = StandardScaler()
df_train[FILM_NUMERIC] = scaler.fit_transform(df_train[FILM_NUMERIC])
df_val[FILM_NUMERIC]   = scaler.transform(df_val[FILM_NUMERIC])

print(f'META_DIM = {META_DIM}')
print(f'Train: {len(df_train):,} | Val: {len(df_val):,}')
print(f'Train malignant rate: {df_train["target"].mean()*100:.3f}%')
print(f'Val   malignant rate: {df_val["target"].mean()*100:.3f}%')

In [ ]:
# ─────────────────────────────────────────────
# CELL 5 — Dataset & DataLoaders
# ─────────────────────────────────────────────
class ISIC2024Dataset(Dataset):
    def __init__(self, df, hdf5_path, meta_cols, transform=None, img_size=224):
        self.df        = df.reset_index(drop=True)
        self.hdf5_path = hdf5_path
        self.meta_cols = meta_cols
        self.transform = transform
        self.img_size  = img_size
        self._hdf5     = None

    def _get_hdf5(self):
        if self._hdf5 is None:
            self._hdf5 = h5py.File(self.hdf5_path, 'r')
        return self._hdf5

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row     = self.df.iloc[idx]
        hf      = self._get_hdf5()
        img     = Image.open(io.BytesIO(hf[row['isic_id']][()])).convert('RGB')
        img     = img.resize((self.img_size, self.img_size), Image.LANCZOS)
        if self.transform:
            img = self.transform(img)
        meta  = torch.tensor(row[self.meta_cols].values.astype(np.float32))
        label = torch.tensor(float(row['target']))
        return img, meta, label

train_tfm = T.Compose([
    T.RandomResizedCrop(CFG['img_size'], scale=(0.7, 1.0)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.5),
    T.RandomRotation(30),
    T.ColorJitter(0.2, 0.2, 0.2, 0.1),
    T.RandAugment(num_ops=2, magnitude=9),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
val_tfm = T.Compose([
    T.Resize((CFG['img_size'], CFG['img_size'])),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_ds = ISIC2024Dataset(df_train, HDF5_PATH, META_COLS, train_tfm, CFG['img_size'])
val_ds   = ISIC2024Dataset(df_val,   HDF5_PATH, META_COLS, val_tfm,   CFG['img_size'])

train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,
                          num_workers=CFG['num_workers'], pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG['batch_size'], shuffle=False,
                          num_workers=CFG['num_workers'], pin_memory=True)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')
imgs, metas, labels = next(iter(train_loader))
print(f'Batch — imgs: {imgs.shape} | metas: {metas.shape} | labels: {labels.shape}')

In [ ]:
# ─────────────────────────────────────────────
# CELL 6 — Novel ACAG Block
# Asymmetry-Aware Cross-Attention Gate
# ─────────────────────────────────────────────
class ACACrossAttentionGate(nn.Module):
    """
    Novel ACAG Block.
    Injects asymmetry residual A = |F - F_h| + |F - F_v|
    as a bias into cross-attention queries.
    CNN tokens attend to ViT tokens guided by asymmetry signal.
    """
    def __init__(self, cnn_dim, vit_dim, d_model=512, n_heads=8, asym_lambda=0.5):
        super().__init__()
        self.d_model     = d_model
        self.n_heads     = n_heads
        self.d_k         = d_model // n_heads
        self.asym_lambda = asym_lambda

        self.proj_q    = nn.Linear(cnn_dim, d_model)
        self.proj_k    = nn.Linear(vit_dim, d_model)
        self.proj_v    = nn.Linear(vit_dim, d_model)
        self.asym_proj = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(cnn_dim, d_model), nn.ReLU(),
        )
        self.out_proj  = nn.Linear(d_model, d_model)
        self.norm      = nn.LayerNorm(d_model)
        self.dropout   = nn.Dropout(0.1)
        self.pool      = nn.AdaptiveAvgPool1d(1)

    def compute_asymmetry_map(self, feat):
        """A = |F - F_h| + |F - F_v| — deviation from both symmetry axes."""
        return (feat - torch.flip(feat, [-1])).abs() + \
               (feat - torch.flip(feat, [-2])).abs()

    def forward(self, cnn_feat, vit_feat):
        B, C, H, W = cnn_feat.shape
        A          = self.compute_asymmetry_map(cnn_feat)
        asym_bias  = self.asym_proj(A).unsqueeze(1)           # B×1×d_model

        cnn_tokens = cnn_feat.flatten(2).permute(0,2,1)       # B×(H*W)×C
        Q = self.proj_q(cnn_tokens) + self.asym_lambda * asym_bias
        K = self.proj_k(vit_feat)
        V = self.proj_v(vit_feat)

        def split(x):
            return x.view(B, -1, self.n_heads, self.d_k).transpose(1,2)

        attn = F.softmax(torch.matmul(split(Q), split(K).transpose(-2,-1)) / self.d_k**0.5, dim=-1)
        attn = self.dropout(attn)
        out  = torch.matmul(attn, split(V))
        out  = out.transpose(1,2).contiguous().view(B, H*W, self.d_model)
        out  = self.norm(self.out_proj(out))
        fused = self.pool(out.transpose(1,2)).squeeze(-1)     # B×d_model
        return fused, attn

print('✅ ACAG Block defined')

In [ ]:
# ─────────────────────────────────────────────
# CELL 7 — FiLM Layer
# Feature-wise Linear Modulation by clinical metadata
# ─────────────────────────────────────────────
class FiLMLayer(nn.Module):
    """FiLM(x; m) = γ(m) ⊙ x + β(m)  — affine modulation by metadata."""
    def __init__(self, meta_dim, feat_dim, hidden_dim=128):
        super().__init__()
        self.feat_dim = feat_dim
        self.film_net = nn.Sequential(
            nn.Linear(meta_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, 2 * feat_dim),
        )
        # Identity initialisation: γ=1, β=0
        nn.init.zeros_(self.film_net[-1].weight)
        nn.init.constant_(self.film_net[-1].bias[:feat_dim], 1.0)
        nn.init.zeros_(self.film_net[-1].bias[feat_dim:])

    def forward(self, x, metadata):
        params = self.film_net(metadata)
        gamma, beta = params[:, :self.feat_dim], params[:, self.feat_dim:]
        return gamma * x + beta

print('✅ FiLM Layer defined')

In [ ]:
# ─────────────────────────────────────────────
# CELL 8 — DermoViT Full Model
# ─────────────────────────────────────────────
class DermoViT(nn.Module):
    """
    DermoViT: Dual-Stream Dermoscopy Vision Transformer
    Stream 1: EfficientNet-B2  (local: texture, borders)
    Stream 2: DeiT-Small       (global: shape, symmetry)
    Fusion:   ACAG Block       (asymmetry-aware cross-attention)
    Cond:     FiLM Layer       (metadata conditioning)
    """
    def __init__(self, meta_dim, d_model=512, n_heads=8,
                 asym_lambda=0.5, drop_rate=0.3, freeze_backbone=True):
        super().__init__()

        # CNN stream
        self.cnn_backbone = timm.create_model(
            'efficientnet_b2', pretrained=True, features_only=True, out_indices=[4]
        )
        cnn_dim = self.cnn_backbone.feature_info[4]['num_chs']   # 1408

        # ViT stream
        self.vit_backbone = timm.create_model(
            'deit_small_patch16_224', pretrained=True, num_classes=0
        )
        vit_dim = self.vit_backbone.embed_dim   # 384

        if freeze_backbone:
            for p in list(self.cnn_backbone.parameters()) + \
                     list(self.vit_backbone.parameters()):
                p.requires_grad = False

        # Novel ACAG fusion
        self.acag = ACACrossAttentionGate(cnn_dim, vit_dim, d_model, n_heads, asym_lambda)

        # Metadata conditioning
        self.film = FiLMLayer(meta_dim, d_model)

        # Classification head
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 256), nn.GELU(), nn.Dropout(drop_rate),
            nn.Linear(256, 1),
        )

    def forward(self, image, metadata):
        cnn_feats        = self.cnn_backbone(image)[0]          # B×1408×7×7
        vit_feats        = self.vit_backbone.forward_features(image)  # B×197×384
        fused, acag_attn = self.acag(cnn_feats, vit_feats)      # B×d_model
        fused            = self.film(fused, metadata)            # B×d_model
        logit            = self.head(fused)                      # B×1
        return logit, acag_attn

    def unfreeze_backbone(self):
        for p in list(self.cnn_backbone.parameters()) + \
                 list(self.vit_backbone.parameters()):
            p.requires_grad = True
        print('✅ Backbone unfrozen — full end-to-end training')

# Instantiate
model = DermoViT(
    meta_dim=META_DIM, d_model=CFG['d_model'], n_heads=CFG['n_heads'],
    asym_lambda=CFG['asym_lambda'], drop_rate=CFG['drop_rate'], freeze_backbone=True,
).to(device)

# Quick sanity check
with torch.no_grad():
    dummy_img  = torch.randn(2, 3, 224, 224).to(device)
    dummy_meta = torch.randn(2, META_DIM).to(device)
    out, _     = model(dummy_img, dummy_meta)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'✅ DermoViT ready | Output shape: {out.shape}')
print(f'   Total params:     {total/1e6:.1f}M')
print(f'   Trainable (warm): {trainable/1e6:.1f}M  (ACAG + FiLM + Head only)')

In [ ]:
# ─────────────────────────────────────────────
# CELL 9 — Loss, Optimizer, Metric
# ─────────────────────────────────────────────
class FocalLoss(nn.Module):
    """FL = -α(1-p_t)^γ log(p_t)  — down-weights easy negatives."""
    def __init__(self, alpha=0.25, gamma=2.0, label_smoothing=0.05):
        super().__init__()
        self.alpha, self.gamma, self.ls = alpha, gamma, label_smoothing

    def forward(self, logits, targets):
        logits  = logits.view(-1)
        targets = targets.view(-1) * (1 - self.ls) + self.ls * 0.5
        bce     = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        p_t     = torch.exp(-bce)
        return (self.alpha * (1 - p_t) ** self.gamma * bce).mean()

class SAM(torch.optim.Optimizer):
    """Sharpness-Aware Minimization — seeks flat loss-landscape minima."""
    def __init__(self, params, base_optimizer_class, rho=0.05, **kwargs):
        super().__init__(params, dict(rho=rho, **kwargs))
        self.base_optimizer = base_optimizer_class(self.param_groups, **kwargs)
        self.param_groups   = self.base_optimizer.param_groups

    @torch.no_grad()
    def first_step(self, zero_grad=False):
        norm = torch.stack([p.grad.norm(2) for g in self.param_groups
                            for p in g['params'] if p.grad is not None]).norm(2)
        for g in self.param_groups:
            scale = g['rho'] / (norm + 1e-12)
            for p in g['params']:
                if p.grad is None: continue
                e_w = p.grad * scale
                p.add_(e_w)
                self.state[p]['e_w'] = e_w
        if zero_grad: self.zero_grad()

    @torch.no_grad()
    def second_step(self, zero_grad=False):
        for g in self.param_groups:
            for p in g['params']:
                if p.grad is None: continue
                p.sub_(self.state[p]['e_w'])
        self.base_optimizer.step()
        if zero_grad: self.zero_grad()

    def step(self, closure=None): raise NotImplementedError

def compute_pauc(y_true, y_pred, min_tpr=0.80):
    """Normalized pAUC above min_tpr — official ISIC 2024 metric."""
    max_fpr = 1 - min_tpr
    return roc_auc_score(y_true, y_pred, max_fpr=max_fpr) / max_fpr

def mixup_batch(imgs, metas, labels, alpha=0.4):
    lam  = np.random.beta(alpha, alpha)
    perm = torch.randperm(imgs.size(0))
    return (lam*imgs + (1-lam)*imgs[perm],
            lam*metas + (1-lam)*metas[perm],
            labels, labels[perm], lam)

criterion = FocalLoss(CFG['focal_alpha'], CFG['focal_gamma'], CFG['label_smoothing'])
optimizer = SAM(model.parameters(), torch.optim.AdamW,
                rho=CFG['sam_rho'], lr=CFG['lr_warmup'], weight_decay=CFG['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer.base_optimizer,
    T_max=CFG['n_warmup'] + CFG['n_finetune'], eta_min=1e-6
)

print('✅ Loss (Focal), Optimizer (SAM+AdamW), Scheduler (Cosine), Metric (pAUC) ready')

In [ ]:
# ─────────────────────────────────────────────
# CELL 10 — Training & Validation Functions
# ─────────────────────────────────────────────
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, all_preds, all_labels = 0, [], []
    t0 = time.time()

    for i, (imgs, metas, labels) in enumerate(loader):
        imgs   = imgs.to(device, non_blocking=True)
        metas  = metas.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        imgs_m, metas_m, la, lb, lam = mixup_batch(
            imgs, metas, labels, CFG['mixup_alpha']
        )

        # SAM Step 1
        logits, _ = model(imgs_m, metas_m)
        loss = lam * criterion(logits.squeeze(), la) + (1-lam) * criterion(logits.squeeze(), lb)
        loss.backward()
        optimizer.first_step(zero_grad=True)

        # SAM Step 2
        logits2, _ = model(imgs_m, metas_m)
        loss2 = lam * criterion(logits2.squeeze(), la) + (1-lam) * criterion(logits2.squeeze(), lb)
        loss2.backward()
        optimizer.second_step(zero_grad=True)

        total_loss += loss.item()
        with torch.no_grad():
            all_preds.extend(torch.sigmoid(logits.squeeze()).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

        if (i + 1) % 100 == 0:
            elapsed = time.time() - t0
            print(f'  Step {i+1}/{len(loader)} | Loss: {loss.item():.4f} | '
                  f'Elapsed: {elapsed:.0f}s', flush=True)

    pauc = compute_pauc(np.array(all_labels), np.array(all_preds))
    return total_loss / len(loader), pauc


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss, all_preds, all_labels = 0, [], []

    for imgs, metas, labels in loader:
        imgs   = imgs.to(device, non_blocking=True)
        metas  = metas.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        logits, _ = model(imgs, metas)
        loss      = criterion(logits.squeeze(), labels)
        total_loss += loss.item()
        all_preds.extend(torch.sigmoid(logits.squeeze()).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    pauc = compute_pauc(np.array(all_labels), np.array(all_preds))
    return total_loss / len(loader), pauc

print('✅ Train and validation functions ready')

In [ ]:
# ─────────────────────────────────────────────
# CELL 11 — TRAINING LOOP  ← RUNS ON KAGGLE GPU
# ─────────────────────────────────────────────
N_EPOCHS = CFG['n_warmup'] + CFG['n_finetune']   # 10 total

history   = {'train_loss': [], 'val_loss': [], 'train_pauc': [], 'val_pauc': []}
best_pauc = 0.0

print(f'Starting training | {N_EPOCHS} epochs | Device: {device}')
print(f'Warmup: {CFG["n_warmup"]} epochs (backbone frozen, lr={CFG["lr_warmup"]})')
print(f'Fine-tune: {CFG["n_finetune"]} epochs (full model, lr={CFG["lr_finetune"]})')
print('─' * 65)

for epoch in range(1, N_EPOCHS + 1):
    t_epoch = time.time()

    # Unfreeze backbone after warmup phase
    if epoch == CFG['n_warmup'] + 1:
        model.unfreeze_backbone()
        for pg in optimizer.base_optimizer.param_groups:
            pg['lr'] = CFG['lr_finetune']
        print(f'\n[Epoch {epoch}] Switching to fine-tune phase (lr={CFG["lr_finetune"]})')

    phase = 'Warmup' if epoch <= CFG['n_warmup'] else 'Finetune'
    print(f'\n[Epoch {epoch}/{N_EPOCHS}] [{phase}]')

    train_loss, train_pauc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss,   val_pauc   = validate(model, val_loader, criterion, device)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_pauc'].append(train_pauc)
    history['val_pauc'].append(val_pauc)

    elapsed = time.time() - t_epoch
    current_lr = optimizer.base_optimizer.param_groups[0]['lr']

    print(f'  Train  → Loss: {train_loss:.4f} | pAUC: {train_pauc:.4f}')
    print(f'  Val    → Loss: {val_loss:.4f}   | pAUC: {val_pauc:.4f}')
    print(f'  LR: {current_lr:.2e} | Time: {elapsed:.0f}s')

    if val_pauc > best_pauc:
        best_pauc = val_pauc
        torch.save({
            'epoch':       epoch,
            'state_dict':  model.state_dict(),
            'val_pauc':    val_pauc,
            'cfg':         CFG,
            'meta_cols':   META_COLS,
            'meta_dim':    META_DIM,
        }, CKPT_PATH)
        print(f'  ✅ Saved best checkpoint (val pAUC: {best_pauc:.4f})')

print(f'\n🏆 Training complete! Best Val pAUC: {best_pauc:.4f}')
print(f'   Checkpoint saved to: {CKPT_PATH}')

In [ ]:
# ─────────────────────────────────────────────
# CELL 12 — Training Curve Plots
# ─────────────────────────────────────────────
epochs = list(range(1, len(history['train_pauc']) + 1))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('DermoViT Training — ISIC 2024', fontsize=14, fontweight='bold')

ax1.plot(epochs, history['train_pauc'], 'b-o', label='Train pAUC', linewidth=2)
ax1.plot(epochs, history['val_pauc'],   'r-o', label='Val pAUC',   linewidth=2)
ax1.axvline(CFG['n_warmup'] + 0.5, color='gray', linestyle='--', alpha=0.7, label='Backbone Unfrozen')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('pAUC (>80% TPR)')
ax1.set_title('Partial AUC — Competition Metric'); ax1.legend(); ax1.grid(True, alpha=0.4)

ax2.plot(epochs, history['train_loss'], 'b-o', label='Train Loss', linewidth=2)
ax2.plot(epochs, history['val_loss'],   'r-o', label='Val Loss',   linewidth=2)
ax2.axvline(CFG['n_warmup'] + 0.5, color='gray', linestyle='--', alpha=0.7)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Focal Loss')
ax2.set_title('Focal Loss Curves'); ax2.legend(); ax2.grid(True, alpha=0.4)

plt.tight_layout()
plot_path = os.path.join(OUT_DIR, 'training_curves.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Plot saved: {plot_path}')

In [ ]:
# ─────────────────────────────────────────────
# CELL 13 — Quick Grad-CAM++ Visualisation
# (runs after training to validate model attention)
# ─────────────────────────────────────────────
class GradCAMPlusPlus:
    def __init__(self, model, target_layer):
        self.model = model
        self.gradients = None
        self.activations = None
        target_layer.register_forward_hook(lambda m,i,o: setattr(self, 'activations', o.detach()))
        target_layer.register_full_backward_hook(lambda m,gi,go: setattr(self, 'gradients', go[0].detach()))

    def __call__(self, img_t, meta_t):
        self.model.eval()
        img_t.requires_grad_(True)
        logit, _ = self.model(img_t.to(device), meta_t.to(device))
        self.model.zero_grad()
        logit.squeeze().backward()
        weights = self.gradients.mean(dim=[2,3], keepdim=True)
        cam = F.relu((weights * self.activations).sum(1)).squeeze()
        cam = cam - cam.min(); cam = cam / (cam.max() + 1e-8)
        cam = F.interpolate(cam.unsqueeze(0).unsqueeze(0),
                            size=(224,224), mode='bilinear', align_corners=False)
        return cam.squeeze().cpu().detach().numpy()

# Load best checkpoint
ckpt = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(ckpt['state_dict'])
print(f'Loaded best checkpoint | Val pAUC: {ckpt["val_pauc"]:.4f} (epoch {ckpt["epoch"]})')

# Sample one malignant + one benign
target_layer = list(model.cnn_backbone.children())[-1]
grad_cam = GradCAMPlusPlus(model, target_layer)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Grad-CAM++ Validation — Does DermoViT look at the lesion?',
             fontsize=13, fontweight='bold')

for row_idx, (label_name, target_val) in enumerate([('MALIGNANT', 1), ('BENIGN', 0)]):
    sample_row = df_val[df_val['target'] == target_val].sample(1, random_state=row_idx)
    isic_id    = sample_row['isic_id'].values[0]
    meta_t     = torch.tensor(sample_row[META_COLS].values.astype(np.float32))

    with h5py.File(HDF5_PATH, 'r') as hf:
        pil_img = Image.open(io.BytesIO(hf[isic_id][()])).convert('RGB').resize((224,224))

    img_t   = val_tfm(pil_img).unsqueeze(0)
    cam_map = grad_cam(img_t, meta_t)
    img_np  = np.array(pil_img) / 255.0

    color   = '#E74C3C' if label_name == 'MALIGNANT' else '#4CAF93'
    axes[row_idx,0].imshow(img_np);                           axes[row_idx,0].set_title(f'{label_name}\nOriginal', color=color, fontweight='bold')
    axes[row_idx,1].imshow(cam_map, cmap='jet');              axes[row_idx,1].set_title('Grad-CAM++ Heatmap')
    axes[row_idx,2].imshow(img_np)
    axes[row_idx,2].imshow(cam_map, cmap='jet', alpha=0.5);   axes[row_idx,2].set_title('Overlay')
    for ax in axes[row_idx]: ax.axis('off')

plt.tight_layout()
cam_path = os.path.join(OUT_DIR, 'gradcam_validation.png')
plt.savefig(cam_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Grad-CAM saved: {cam_path}')

In [ ]:
# ─────────────────────────────────────────────
# CELL 14 — Final Summary
# ─────────────────────────────────────────────
print('=' * 65)
print('  DERMOVIT — ISIC 2024 TRAINING COMPLETE')
print('=' * 65)
print(f'  Best Val pAUC:  {best_pauc:.4f}  (target > 0.75)')
print(f'  Checkpoint:     {CKPT_PATH}')
print(f'  Training plot:  {OUT_DIR}/training_curves.png')
print(f'  Grad-CAM plot:  {OUT_DIR}/gradcam_validation.png')
print('─' * 65)
print('  Architecture:   EfficientNet-B2 + DeiT-Small + ACAG + FiLM')
print(f'  Epochs:         {N_EPOCHS}  (Warmup: {CFG["n_warmup"]} | Fine-tune: {CFG["n_finetune"]})')
print(f'  Loss:           Focal Loss  (α={CFG["focal_alpha"]}, γ={CFG["focal_gamma"]})')
print(f'  Optimizer:      SAM + AdamW (ρ={CFG["sam_rho"]})')
print(f'  Augmentation:   RandAugment + MixUp (α={CFG["mixup_alpha"]})')
print(f'  Metric:         pAUC > 80% TPR (ISIC 2024 official)')
print('=' * 65)